# ARC-05: Reproducible Numerical Orbital Validation

Required cases include circular and elliptical motion, energy and angular-momentum drift, timestep sensitivity, RK4 convergence evidence, invalid inputs, collision detection, zero radius, and non-finite values.

Each tolerance is specific to this educational experiment and must not be treated as a universal aerospace tolerance.


In [1]:
import math
import numpy as np

MU = 3.986004418e14


def strict_derivative(time_s, state, mu):
    if isinstance(state, (str, bytes, bytearray, bool, np.bool_)):
        raise ValueError("State must be numeric")
    try:
        components = list(state)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State must be numeric") from exc
    for component in components:
        if isinstance(component, (bool, np.bool_, str, bytes, bytearray, complex, np.complexfloating, list, tuple, np.ndarray)):
            raise ValueError("State must contain only real numeric values")
    if isinstance(time_s, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Time must be numeric")
    time_s = float(time_s)
    if not math.isfinite(time_s) or time_s < 0.0:
        raise ValueError("Time must be finite and non-negative")
    y = np.asarray(state, dtype=np.float64)
    if y.shape != (6,) or not np.all(np.isfinite(y)):
        raise ValueError("State must be finite with six components")
    if isinstance(mu, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("mu must be numeric")
    mu = float(mu)
    if not math.isfinite(mu) or mu <= 0:
        raise ValueError("mu must be finite and positive")
    r = float(np.hypot(np.hypot(abs(y[0]), abs(y[1])), abs(y[2])))
    if not math.isfinite(r) or r <= 0:
        raise ValueError("Zero-radius or non-finite position is invalid")
    position_unit = y[:3] / r
    acceleration_scale = (mu / r) / r
    if not math.isfinite(acceleration_scale):
        raise ValueError("Acceleration magnitude is non-finite")
    if acceleration_scale == 0.0:
        raise ValueError("Acceleration magnitude underflows float64")
    acceleration = -acceleration_scale * position_unit
    result = np.concatenate((y[3:], acceleration)).astype(np.float64)
    if not np.all(np.isfinite(result)):
        raise ValueError("Derivative contains non-finite values")
    return result


def rk4(state, time_s, dt, derivative, *args):
    if isinstance(time_s, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Time must be numeric")
    time_s = float(time_s)
    if not math.isfinite(time_s) or time_s < 0.0:
        raise ValueError("Time must be finite and non-negative")
    if isinstance(dt, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Timestep must be numeric")
    dt = float(dt)
    if not math.isfinite(dt) or dt <= 0:
        raise ValueError("Timestep must be finite and positive")
    y = np.asarray(state, dtype=np.float64).copy()
    if y.shape != (6,) or not np.all(np.isfinite(y)):
        raise ValueError("Invalid state")
    k1 = derivative(time_s, y, *args)
    k2 = derivative(time_s + 0.5*dt, y + 0.5*dt*k1, *args)
    k3 = derivative(time_s + 0.5*dt, y + 0.5*dt*k2, *args)
    k4 = derivative(time_s + dt, y + dt*k3, *args)
    stages = (k1, k2, k3, k4)
    if any(k.shape != y.shape or not np.all(np.isfinite(k)) for k in stages):
        raise ValueError("RK4 intermediate stage is invalid")
    result = y + dt/6.0*(k1 + 2*k2 + 2*k3 + k4)
    if not np.all(np.isfinite(result)):
        raise ValueError("RK4 produced a non-finite state")
    return result


def linear_radius_collision_event(previous_time, previous_state, next_time, next_state, collision_radius):
    previous_state = np.asarray(previous_state, dtype=np.float64)
    next_state = np.asarray(next_state, dtype=np.float64)
    if previous_state.shape != (6,) or next_state.shape != (6,) or not np.all(np.isfinite(previous_state)) or not np.all(np.isfinite(next_state)):
        raise ValueError("Collision bracket states must be finite and have shape (6,)")
    previous_radius = float(np.hypot(np.hypot(abs(previous_state[0]), abs(previous_state[1])), abs(previous_state[2])))
    next_radius = float(np.hypot(np.hypot(abs(next_state[0]), abs(next_state[1])), abs(next_state[2])))
    if not math.isfinite(previous_radius) or not math.isfinite(next_radius):
        raise ValueError("Collision bracket contains a non-finite radius")
    if isinstance(collision_radius, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Collision threshold must be numeric")
    if isinstance(previous_time, (bool, np.bool_, str, bytes, bytearray)) or isinstance(next_time, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Collision bracket times must be numeric")
    try:
        previous_time = float(previous_time)
        next_time = float(next_time)
        collision_radius = float(collision_radius)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("Collision bracket times and threshold must be numeric") from exc
    if not math.isfinite(previous_time) or not math.isfinite(next_time) or next_time < previous_time:
        raise ValueError("Collision bracket times must be finite and ordered")
    if not math.isfinite(collision_radius) or collision_radius <= 0:
        raise ValueError("Collision threshold must be finite and positive")
    if previous_radius == collision_radius:
        return float(previous_time), previous_state.copy(), "exact_sample"
    if next_radius == collision_radius:
        return float(next_time), next_state.copy(), "exact_sample"
    if previous_radius == next_radius:
        raise ValueError("Invalid collision interval: endpoint radii are degenerate and do not bracket the threshold")
    if not (min(previous_radius, next_radius) <= collision_radius <= max(previous_radius, next_radius)):
        raise ValueError("Invalid collision interval: threshold is not bracketed by endpoint radii")
    if previous_radius != next_radius:
        alpha = (collision_radius - previous_radius)/(next_radius - previous_radius)
        if not math.isfinite(alpha) or alpha < 0.0 or alpha > 1.0:
            raise ValueError("Invalid collision interval: interpolation fraction is outside [0, 1]")
    else:
        alpha = 0.0
    event_time = float(previous_time) + alpha*(float(next_time)-float(previous_time))
    event_state = ((1.0-alpha)*previous_state + alpha*next_state).astype(np.float64)
    return event_time, event_state, "linear_radius_interpolation"


def propagate(initial, duration, nominal_dt, mu=MU, collision_radius=None):
    y = np.asarray(initial, dtype=np.float64).copy()
    if y.shape != (6,) or not np.all(np.isfinite(y)):
        raise ValueError("Initial state is invalid")
    times = [0.0]
    states = [y.copy()]
    t = 0.0
    collision_index = None
    previous_state = y.copy()
    previous_time = 0.0
    globals()["collision_event_details"] = None
    if collision_radius is not None and np.linalg.norm(y[:3]) <= collision_radius:
        collision_index = 0
    collision_event_details = None
    while t < duration and collision_index is None:
        dt = min(nominal_dt, duration-t)
        previous_time = t
        previous_state = y.copy()
        y = rk4(y, t, dt, strict_derivative, mu)
        t += dt
        if abs(t-duration) <= 1e-12*max(1.0, duration):
            t = duration
        times.append(t)
        states.append(y.copy())
        if collision_radius is not None and np.linalg.norm(y[:3]) <= collision_radius:
            collision_index = len(states)-1
            collision_event_details = linear_radius_collision_event(
                previous_time, previous_state, t, y, collision_radius
            )
            globals()["collision_event_details"] = collision_event_details
    return np.asarray(times, dtype=np.float64), np.vstack(states).astype(np.float64), collision_index


def orbital_energy(state, mu=MU):
    r = float(np.linalg.norm(state[:3]))
    speed = float(np.linalg.norm(state[3:]))
    if r <= 0:
        raise ValueError("Position radius must be positive")
    return 0.5*speed*speed - mu/r


def angular_momentum_magnitude(state):
    return float(np.linalg.norm(np.cross(state[:3], state[3:])))


In [2]:
circular_radius = 7_000_000.0
circular_speed = math.sqrt(MU/circular_radius)
circular_initial = np.array([circular_radius, 0.0, 0.0, 0.0, circular_speed, 0.0], dtype=np.float64)
period = 2*math.pi*math.sqrt(circular_radius**3/MU)

times, states, collision = propagate(circular_initial, period, 20.0)
position_error = float(np.linalg.norm(states[-1,:3] - circular_initial[:3]))
print("Circular period (analytic):", period)
print("Return-position error with 20 s RK4 steps:", position_error, "m")
assert collision is None
assert position_error < 10_000.0

semi_major_axis = 8_000_000.0
periapsis_radius = 7_000_000.0
elliptical_speed = math.sqrt(MU*(2.0/periapsis_radius - 1.0/semi_major_axis))
elliptical_initial = np.array([periapsis_radius, 0.0, 0.0, 0.0, elliptical_speed, 0.0], dtype=np.float64)
elliptical_energy = orbital_energy(elliptical_initial)
elliptical_h = angular_momentum_magnitude(elliptical_initial)
analytic_eccentricity = math.sqrt(max(0.0, 1.0 + 2.0*elliptical_energy*elliptical_h**2/MU**2))
analytic_semi_major_axis = -MU/(2.0*elliptical_energy)
apoapsis_radius = analytic_semi_major_axis*(1.0 + analytic_eccentricity)
derived_periapsis = analytic_semi_major_axis*(1.0 - analytic_eccentricity)
elliptical_period = 2.0*math.pi*math.sqrt(analytic_semi_major_axis**3/MU)
eccentricity_estimate = analytic_eccentricity
assert math.isclose(derived_periapsis, periapsis_radius, rel_tol=1e-12)
assert math.isclose(analytic_semi_major_axis, semi_major_axis, rel_tol=1e-12)
assert 0.0 < analytic_eccentricity < 1.0

e_times, e_states, _ = propagate(elliptical_initial, elliptical_period, 10.0)
elliptical_radii = np.linalg.norm(e_states[:,:3], axis=1)
numerical_periapsis = float(np.min(elliptical_radii))
numerical_apoapsis = float(np.max(elliptical_radii))
periapsis_index = int(np.argmin(elliptical_radii))
apoapsis_index = int(np.argmax(elliptical_radii))
print("Elliptical analytical periapsis/apoapsis:", derived_periapsis, apoapsis_radius)
print("Elliptical numerical periapsis/apoapsis:", numerical_periapsis, numerical_apoapsis)
print("Elliptical period:", elliptical_period)
assert np.all(np.diff(e_times) > 0.0)
assert math.isclose(numerical_periapsis, derived_periapsis, rel_tol=2e-7)
assert math.isclose(numerical_apoapsis, apoapsis_radius, rel_tol=2e-7)

elliptical_drift = {}
elliptical_position_errors = {}
for nominal_dt in (60.0, 30.0, 15.0):
    _, elliptical_states, elliptical_collision = propagate(elliptical_initial, elliptical_period, nominal_dt)
    assert elliptical_collision is None
    elliptical_energies = np.asarray([orbital_energy(state) for state in elliptical_states])
    elliptical_angular_momenta = np.asarray([angular_momentum_magnitude(state) for state in elliptical_states])
    energy_absolute = float(np.max(np.abs(elliptical_energies-elliptical_energies[0])))
    angular_absolute = float(np.max(np.abs(elliptical_angular_momenta-elliptical_angular_momenta[0])))
    energy_relative = energy_absolute/abs(elliptical_energies[0])
    angular_relative = angular_absolute/elliptical_angular_momenta[0]
    return_error = float(np.linalg.norm(elliptical_states[-1,:3]-elliptical_initial[:3]))
    elliptical_drift[nominal_dt] = (energy_absolute, energy_relative, angular_absolute, angular_relative)
    elliptical_position_errors[nominal_dt] = return_error
print("Elliptical energy/angular drift by timestep:", elliptical_drift)
print("Elliptical one-period position errors:", elliptical_position_errors)
elliptical_error_ratios = [elliptical_position_errors[60.0]/elliptical_position_errors[30.0], elliptical_position_errors[30.0]/elliptical_position_errors[15.0]]
print("Elliptical nonlinear return-error ratios:", elliptical_error_ratios)
assert elliptical_position_errors[60.0] > elliptical_position_errors[30.0] > elliptical_position_errors[15.0]
assert all(12.0 < ratio < 64.0 for ratio in elliptical_error_ratios)

energies = np.asarray([orbital_energy(state) for state in states], dtype=np.float64)
angular_momenta = np.asarray([angular_momentum_magnitude(state) for state in states], dtype=np.float64)
max_energy_drift = float(np.max(np.abs(energies - energies[0])))
max_angular_drift = float(np.max(np.abs(angular_momenta - angular_momenta[0])))
print("Measured max energy drift:", max_energy_drift, "m^2/s^2")
print("Measured max angular-momentum drift:", max_angular_drift, "m^2/s")
assert max_energy_drift/abs(energies[0]) < 1e-6
assert max_angular_drift/angular_momenta[0] < 1e-9

timestep_errors = {}
for nominal_dt in (60.0, 30.0, 15.0):
    _, trial_states, _ = propagate(circular_initial, period, nominal_dt)
    timestep_errors[nominal_dt] = float(np.linalg.norm(trial_states[-1,:3] - circular_initial[:3]))
print("Circular timestep sensitivity:", timestep_errors)
assert timestep_errors[60.0] > timestep_errors[30.0] > timestep_errors[15.0]
circular_error_ratios = [timestep_errors[60.0]/timestep_errors[30.0], timestep_errors[30.0]/timestep_errors[15.0]]
print("Circular nonlinear return-error ratios:", circular_error_ratios)
assert all(12.0 < ratio < 64.0 for ratio in circular_error_ratios)

for invalid_state in ([1,2,3], [0,0,0,0,0,0], [1,2,3,4,5,float("nan")], [1,2,3,4,5,float("inf")]):
    try:
        strict_derivative(0.0, invalid_state, MU)
        raise AssertionError("Invalid state accepted")
    except ValueError:
        pass

_times, _states, collision_index = propagate(np.array([6378137.0,0,0,0,0,0], dtype=np.float64), 100.0, 1.0, collision_radius=6378137.0)
assert collision_index == 0
try:
    strict_derivative(0.0, np.array([0.0,0,0,0,0,0], dtype=np.float64), MU)
    raise AssertionError("Zero radius accepted")
except ValueError:
    pass

print("Elliptical eccentricity estimate:", eccentricity_estimate)
print("Numerical validation cases: PASSED")


Circular period (analytic): 5828.516637686015
Return-position error with 20 s RK4 steps: 0.24471202928012176 m
Elliptical analytical periapsis/apoapsis: 7000000.000000002 8999999.999999994
Elliptical numerical periapsis/apoapsis: 6999999.999961766 8999999.90744396
Elliptical period: 7121.081577578021
Elliptical energy/angular drift by timestep: {60.0: (3.4853154830634594, np.float64(1.399021222284439e-07), 2763.742500305176, np.float64(4.9329124543028956e-08)), 30.0: (0.17080063000321388, np.float64(6.85601367552579e-09), 87.10848999023438, np.float64(1.5547705877117657e-09)), 15.0: (0.009233292192220688, np.float64(3.7062847800268286e-10), 2.7242889404296875, np.float64(4.862493101973621e-11))}
Elliptical one-period position errors: {60.0: 16.0254684180572, 30.0: 0.8414829245348426, 15.0: 0.04767541392975655}
Elliptical nonlinear return-error ratios: [19.044318013839444, 17.650248947490148]
Measured max energy drift: 0.023118458688259125 m^2/s^2
Measured max angular-momentum drift: 21